# Perturbations

`HighFidelityEphemerisModel.jl` augments the central two-body term with optional perturbations:

- third-body gravity from other massive bodies
- spherical harmonics of the central body
- solar radiation pressure (SRP)
- atmospheric drag

Mathematical definitions appear in the [Overview](../docs/src/overview.md). This page shows how to enable each term in `SpiceParameters` or `EphemeridesParameters` and propagate with `OrdinaryDiffEq.jl`.

> **Tip**
>
> Before running the examples, download SPICE kernels and set `ENV["SPICE"]` to your kernel directory. Generic kernels: [https://naif.jpl.nasa.gov/pub/naif/generic_kernels/](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/).

> **Note**
>
> The **first** entry in `naif_ids` is always the central body. All positions and velocities in the state vector are expressed in canonical units (`DU`, `DU/TU`) relative to that body.

## Running this notebook

The next cell activates the environment shipped with the examples (`examples/Project.toml`), which uses the local copy of `HighFidelityEphemerisModel.jl`. Skip it if you would rather use your own environment with `HighFidelityEphemerisModel`, `OrdinaryDiffEq`, and `SPICE` installed.

SPICE kernels are loaded from the directory given by the `SPICE` environment variable, which should contain the `lsk/`, `spk/`, `pck/`, and `fk/` subfolders of the [NAIF generic kernels](https://naif.jpl.nasa.gov/pub/naif/generic_kernels/). Edit `spice_dir` below if your kernels live elsewhere.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

## Shared setup

The snippets below assume the following preamble (load once per Julia session):

In [ ]:
using LinearAlgebra
using SPICE
using OrdinaryDiffEq
using HighFidelityEphemerisModel

# load SPICE kernels
spice_dir = ENV["SPICE"]
furnsh(joinpath(spice_dir, "lsk", "naif0012.tls"))
furnsh(joinpath(spice_dir, "spk", "de440.bsp"))
furnsh(joinpath(spice_dir, "pck", "gm_de440.tpc"))
furnsh(joinpath(spice_dir, "pck", "pck00011.tpc"))           # IAU_EARTH frame
furnsh(joinpath(spice_dir, "pck", "earth_latest_high_prec.bpc"))  # ITRF93 frame (Earth gravity & drag)
furnsh(joinpath(spice_dir, "pck", "moon_pa_de440_200625.bpc"))
furnsh(joinpath(spice_dir, "fk", "moon_de440_250416.tf"))

naif_frame = "J2000"
abcorr = "NONE"
et0 = str2et("2026-01-05T00:00:00")

# typical initial state and short propagation window (canonical units)
x0 = [1.05, 0.0, 0.3, 0.5, 1.0, 0.0]

## Third-body perturbations

Third-body accelerations are included automatically for every body listed in `naif_ids` after the central body. Ephemerides are queried from the backend selected by the parameter type.

In [ ]:
naif_ids = ["301", "399", "10"]   # Moon, Earth, Sun (301 = central body)
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 1e5

parameters = SpiceParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr,
)

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_Nbody!, x0, tspan, parameters)
sol_nbody = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)
sol_nbody.u[end]

> **Note**
>
> NAIF body IDs: [https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/req/naif_ids.html](https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/req/naif_ids.html)

## Spherical harmonics

Use the `NbodySH` equations of motion when the gravity field of the central body is not spherical. Provide a gravity model file, the maximum degree `nmax`, and the planet-centered planet-fixed (PCPF) frame name.

The gravity field must belong to the **central body** (`naif_ids[1]`), and `frame_PCPF` must be the body-fixed frame it is expressed in. The repository ships `data/luna/gggrx_1200l_sha_20x20.tab` and `gggrx_1200l_sha_50x50.tab` (Moon, use with `"MOON_PA"`) and `data/earth/GGM03S.tab` (Earth, degree 180, use with `"ITRF93"` or the lower-accuracy `"IAU_EARTH"`).

Accelerations are evaluated with fully-normalized coefficients and the normalized Cunningham recursion (one pass, no factorials), which is stable to high degree (tested to 180 x 180, with ForwardDiff Jacobians and Hessians tested to 100 x 100), so `nmax` is only limited by the file and by run time (roughly quadratic in `nmax`).

In [ ]:
naif_ids = ["301", "399", "10"]
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 1e5
nmax = 4
filepath_spherical_harmonics = joinpath(
    pkgdir(HighFidelityEphemerisModel),
    "data", "luna", "gggrx_1200l_sha_20x20.tab",
)

parameters = SpiceParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    filepath_spherical_harmonics = filepath_spherical_harmonics,
    nmax = nmax,
    frame_PCPF = "MOON_PA",
)

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_NbodySH!, x0, tspan, parameters)
sol_SH = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)

# effect of the spherical harmonics on the final position, in km
norm(sol_SH.u[end][1:3] - sol_nbody.u[end][1:3]) * DU

> **Warning**
>
> Spherical harmonics require `eom_NbodySH!` or a backend-specific `eom_NbodySH_*` function. The `Nbody` variants do not evaluate harmonic terms.

## Ephemerides.jl backend

`EphemeridesParameters` uses Julia-native ephemeris queries instead of direct SPICE ephemeris calls at runtime. Pass either an existing `ephemerides_provider` or a list of ephemeris files through `ephemerides_files`. For spherical harmonics, also provide the binary PCK needed by the body-fixed frame transformation.

In [ ]:
paths = [
    joinpath(spice_dir, "spk", "de440.bsp"),
    joinpath(spice_dir, "pck", "moon_pa_de440_200625.bpc"),
]

parameters = EphemeridesParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    filepath_spherical_harmonics = filepath_spherical_harmonics,
    nmax = nmax,
    frame_PCPF = "MOON_PA",
    include_srp = true,
    ephemerides_files = paths,
)

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_NbodySH!, x0, tspan, parameters)
sol_ephemerides = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)
sol_ephemerides.u[end]

> **Note**
>
> The Ephemerides.jl backend supports third-body gravity, spherical harmonics, SRP, and drag. Frame transforms such as `"J2000"` to `"MOON_PA"` are handled through `FrameTransformations.jl`. The supported body-fixed frames are `"MOON_PA"` (needs a lunar binary PCK such as `moon_pa_de440_200625.bpc`), `"ITRF93"` (needs a binary Earth PCK such as `earth_latest_high_prec.bpc`), and `"IAU_EARTH"` (IAU 2009 rotation model, no kernel needed); other body-fixed frames (e.g. `"IAU_MOON"`) raise an error at construction, unless a custom `ephemerides_frame_system` is passed. For common DE440 body pairs, the helper routines also try segment-chain fallbacks through the solar-system barycenter and Earth-Moon barycenter when a direct pair is unavailable.

An Earth-centred problem with the Ephemerides.jl backend uses the Earth gravity field and the `"ITRF93"` axes from the binary Earth PCK (both drag and spherical harmonics are evaluated in `frame_PCPF`):

In [ ]:
naif_ids_earth = ["399", "301", "10"]   # Earth-centered, with Moon & Sun as third bodies
GMs_earth = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids_earth]
DU_earth = 6378.0

parameters = EphemeridesParameters(
    et0, DU_earth, GMs_earth, naif_ids_earth, naif_frame, abcorr;
    filepath_spherical_harmonics = joinpath(
        pkgdir(HighFidelityEphemerisModel), "data", "earth", "GGM03S.tab",
    ),
    nmax = 8,
    frame_PCPF = "ITRF93",
    include_drag = true,
    f_density = harris_priester_f_density(),
    ephemerides_files = [
        joinpath(spice_dir, "spk", "de440.bsp"),
        joinpath(spice_dir, "pck", "earth_latest_high_prec.bpc"),
    ],
)

# circular orbit at 400 km altitude
r0 = (6378.137 + 400.0) / DU_earth
x0_earth_ephem = [r0, 0.0, 0.0, 0.0, sqrt(1 / r0), 0.0]
tspan = (0.0, 3 * 3600 / parameters.TU)
prob = ODEProblem(eom_NbodySH!, x0_earth_ephem, tspan, parameters)
sol_earth_ephemerides = solve(prob, Vern8(), reltol=1e-12, abstol=1e-12)
sol_earth_ephemerides.u[end]

## Solar radiation pressure

Enable the cannonball SRP model with `include_srp = true` and spacecraft properties `srp_Cr`, `srp_Am`. The Sun (NAIF ID `"10"`) **must** appear in `naif_ids` so its ephemeris is available. No eclipse/shadow model is applied: the spacecraft is always assumed to be in sunlight.

In [ ]:
naif_ids = ["301", "399", "10"]
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 1e5

parameters = SpiceParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    include_srp = true,
    srp_Cr = 1.15,
    srp_Am = 0.002,
    srp_P0 = 4.56e-6,
)

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_Nbody!, x0, tspan, parameters)
sol_SRP = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)

# effect of SRP on the final position, in km
norm(sol_SRP.u[end][1:3] - sol_nbody.u[end][1:3]) * DU

> **Warning**
>
> If `include_srp = true` but `"10"` is missing from `naif_ids`, parameter construction throws an error.

## Atmospheric drag

Drag uses a quadratic law with density supplied by a callback `f_density(et, r_km)` returning $\rho$ in kg/m³. The atmosphere is modeled as co-rotating with angular rate `omega_atm` (default: Earth sidereal rate about the $z$-axis).

When drag is enabled, the EOM transform the spacecraft position from `naif_frame` to `frame_PCPF` before calling `f_density`. Custom callbacks must therefore expect **`r_km` in `frame_PCPF` (km)**, not in the inertial frame. Set `frame_PCPF` to an Earth body-fixed frame: `"ITRF93"` (binary Earth PCK such as `earth_latest_high_prec.bpc`) or `"IAU_EARTH"` (with SPICE this requires a text PCK such as `pck00011.tpc`; the Ephemerides.jl backend needs no kernel for it).

The Harris–Priester model of Montenbruck & Gill (*Satellite Orbits*, §3.5.2) is available via `harris_priester_f_density(R_earth_km=6378.137; flattening, n=6, lag_deg=30, sun_position)`:

- the altitude is the geodetic height above an oblate ellipsoid with equatorial radius `R_earth_km` (in km; WGS-84 by default; this is not the canonical distance unit `DU`) and `flattening` (`flattening = 0` gives `norm(r_km) - R_earth_km`);
- the antapex/apex tables `ρ_min(h)`, `ρ_max(h)` (mean solar activity) are interpolated exponentially, and `ρ = ρ_min + (ρ_max - ρ_min) cos^n(ψ/2)`, where `ψ` is the angle to the apex of the diurnal bulge (Sun declination, 30° east of the Sun ≈ 14h local time). Use `n = 2` for low-inclination and `n = 6` (default) for polar orbits;
- the density is **zero outside the model's validity range 100–1000 km**: a re-entering spacecraft loses all drag below 100 km (geodetic), so stop the propagation there with an altitude event (Jacchia–Roberts throws an error instead);
- the profile is only C⁰ in altitude: `∂ρ/∂h` jumps at each table height, which degrades ForwardDiff STMs on orbits crossing table heights and can make drag-dependent optimization non-smooth;
- `r_km` must be in an **Earth-fixed** `frame_PCPF` (`"ITRF93"` or `"IAU_EARTH"`): the bulge is placed with the Sun in Earth-fixed coordinates, so an inertial `frame_PCPF` silently gives the wrong local time;
- the Sun direction comes from a SPICE-free analytic model (≈ 0.01°), so the callback is thread-safe and ForwardDiff-compatible. Pass `sun_position = et -> r_sun` (Sun position in `frame_PCPF`) to use another source, e.g. `ephemerides_sun_position_function(parameters.ephemerides_frame_system, "ITRF93")`.

Up to v0.2.3 the model had no diurnal bulge and used a spherical altitude; `harris_priester_f_density(6378.0; flattening=0, use_min=true)` approximates that old behaviour (`use_min = true/false` selects a single table column).

The Jacchia–Roberts model is available via `jacchia_roberts_f_density` (constant F10.7 / F10.7a / Kp; analytic geodetic latitude/altitude). With the default `sun_model = :spice` it calls SPICE (`spkpos`, `pxform`, `deltet`) at every evaluation and is therefore **not thread-safe**; use `jacchia_roberts_f_density(sun_model = :analytic)` (or pass `sun_position`) for a SPICE-free callback that can be used with `EnsembleThreads`.

Drag is supported by `eom_Nbody!` and `eom_NbodySH!` for all parameter backends, including Ephemerides.jl.

In [ ]:
naif_ids = ["399", "10"]   # Earth-centered; Sun for optional third-body / SRP
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
DU = 6378.0

# Harris-Priester with diurnal bulge, geodetic altitude (WGS-84), analytic Sun;
# n = 2 is the recommended exponent for this near-equatorial orbit
f_density = harris_priester_f_density(n = 2)

parameters = SpiceParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    frame_PCPF = "IAU_EARTH",
    include_drag = true,
    drag_Cd = 2.2,
    drag_Am = 0.01,
    f_density = f_density,
)

x0_earth = [1.05, 0.0, 0.01, 0.0, 1.0, 0.0]
tspan = (0.0, 2 * 86400 / parameters.TU)
prob = ODEProblem(eom_Nbody!, x0_earth, tspan, parameters)
sol_drag = solve(prob, Vern7(), reltol=1e-12, abstol=1e-12)
sol_drag.u[end]

`f_density` is expected to be a function with signature `f_density(et, r_km)`, where `et` is the epoch in ephemeris seconds and `r_km` is the position vector in **`frame_PCPF`**, km. For multithreaded propagations (`EnsembleThreads`, `Threads.@threads`) the callback must be thread-safe, i.e. it must not call SPICE; for example:

```julia
f_density_jr = jacchia_roberts_f_density(sun_model = :analytic, F107 = 150.0, F107a = 150.0, Kp = 3.0)
```

> **Note**
>
> `include_drag = true` requires both `f_density` and `frame_PCPF` to be provided. Coefficient `k_drag` is computed internally from `drag_Cd` and `drag_Am` in SI units.

## Combining perturbations

Flags compose in a single parameter object. The example below propagates a low Earth orbit with the Julia-native Ephemerides.jl backend (compatible with `EnsembleThreads` and `ForwardDiff`), and enables lunar and solar third-body gravity, Earth spherical harmonics (GGM03S, matching the Earth-centred problem), SRP, and drag together.

In [ ]:
naif_ids = ["399", "301", "10"]     # Earth-centered, with Moon & Sun as third bodies
GMs = [bodvrd(ID, "GM", 1)[1] for ID in naif_ids]
R_earth_km = 6378.0
DU = R_earth_km
nmax = 4
filepath_spherical_harmonics = joinpath(
    pkgdir(HighFidelityEphemerisModel),
    "data", "earth", "GGM03S.tab",
)

parameters = EphemeridesParameters(
    et0, DU, GMs, naif_ids, naif_frame, abcorr;
    ephemerides_files = joinpath(spice_dir, "spk", "de440.bsp"),   # IAU_EARTH needs no PCK
    filepath_spherical_harmonics = filepath_spherical_harmonics,
    nmax = nmax,
    frame_PCPF = "IAU_EARTH",
    include_srp = true,
    srp_Cr = 1.15,
    srp_Am = 0.002,
    include_drag = true,
    drag_Cd = 2.2,
    drag_Am = 0.01,
    f_density = harris_priester_f_density(n = 4),   # intermediate inclination
)

# circular orbit at 400 km altitude, 51.6 deg inclination
r0 = (R_earth_km + 400.0) / DU
v0 = sqrt(1 / r0)
inc = deg2rad(51.6)
x0_leo = [r0, 0.0, 0.0, 0.0, v0 * cos(inc), v0 * sin(inc)]

tspan = (0.0, 6 * 3600 / parameters.TU)
prob = ODEProblem(eom_NbodySH!, x0_leo, tspan, parameters)
sol_combined = solve(prob, Vern8(), reltol=1e-12, abstol=1e-12)

# final altitude, in km
norm(sol_combined.u[end][1:3]) * DU - R_earth_km

> **Note**
>
> `EphemeridesParameters` evaluates ephemerides and frame rotations directly from the loaded kernels at every EOM call, so the propagation only needs to stay within the time span covered by those kernels. For the more accurate `"ITRF93"` Earth-fixed frame, also pass a binary Earth PCK such as `earth_latest_high_prec.bpc` in `ephemerides_files`, as in the Ephemerides.jl backend example above.

## Choosing an equation of motion

| Model | Perturbations included | SPICE | Ephemerides.jl |
|-------|------------------------|-------|----------------|
| `Nbody` | third-body, optional SRP & drag | `SpiceParameters` | `EphemeridesParameters` |
| `NbodySH` | above + spherical harmonics | `SpiceParameters` | `EphemeridesParameters` |

See the full function list and STM options in the [Overview](../docs/src/overview.md).

| Use case | Recommended EOM |
|----------|-----------------|
| High accuracy, few propagations | `SpiceParameters` + `eom_NbodySH!` |
| Julia-native ephemerides, `EnsembleThreads`, sensitivities | `EphemeridesParameters` + `eom_NbodySH!` |
| No harmonics needed | `eom_Nbody!` |

For IVP setup and STM propagation, see [Basics](basics.ipynb). For Jacobians and Hessians, see [Jacobians & Hessians](jacobians_hessians.ipynb).